# Dyck Hidden State Evaluation: Train, Probe, Compression

This notebook is runnable end-to-end with the current `src/hse` implementation. By default it runs a small smoke experiment over both Dyck settings and all four model families:

- Dyck no noise
- Dyck 50% noise
- RNN, LSTM, Transformer, and official Mamba when `mamba-ssm` is available

The default values are intentionally small so `Run All` finishes quickly. For the real experiment, increase `TRAINING_STEPS`, `EXTRACT_EXAMPLES`, `BATCH_SIZE`, and `SEEDS` in Section 1.

## 0. Setup

In [1]:
from pathlib import Path
import importlib.util
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RESULTS = ROOT / "results" / "dyck_all_models_notebook"
RESULTS.mkdir(parents=True, exist_ok=True)

ROOT, RESULTS

(WindowsPath('C:/Users/HP/Desktop/Research/UWM Yiqiao Zhong/Hidden State Evaluation'),
 WindowsPath('C:/Users/HP/Desktop/Research/UWM Yiqiao Zhong/Hidden State Evaluation/results/dyck_all_models_notebook'))

In [2]:
import numpy as np
import pandas as pd
import torch

from hse.analysis.compression import run_compression_probes
from hse.analysis.probes import load_probe_data, run_sufficient_statistic_probes
from hse.models import OfficialMambaUnavailableError, build_model
from hse.tasks.dyck import DyckConfig, DyckSampler
from hse.utils import evaluate_causal_lm, extract_hidden_states, save_json, train_causal_lm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



'cuda'

## 1. Settings

The notebook defaults to a quick smoke run. For the real run, use something like:

```python
TRAINING_STEPS = 10_000
BATCH_SIZE = 128
EXTRACT_EXAMPLES = 50_000
SEEDS = [0, 1, 2]
```

Everything else can stay the same.

In [3]:
DYCK_SETTINGS = {
    "dyck_no_noise": dict(
        dyck_pairs=24,
        total_length=48,
        seq_len=48,
        repeat_prob=1.0,
        n_tasks=512,
        prefix_probe_max_len=7,
    ),
    "dyck_50_noise": dict(
        dyck_pairs=10,
        total_length=20,
        seq_len=48,
        repeat_prob=0.5,
        n_tasks=512,
        prefix_probe_max_len=7,
    ),
}

MAMBA_SSM_AVAILABLE = importlib.util.find_spec("mamba_ssm") is not None
RUN_MAMBA_LIKE_FALLBACK_WHEN_OFFICIAL_UNAVAILABLE = True

MODEL_SPECS = {
    "rnn": dict(layers=3, emb_dim=128, hidden_dim=256, state_kind="h"),
    "lstm": dict(layers=3, emb_dim=128, hidden_dim=128, state_kind="c"),
    "transformer": dict(layers=3, emb_dim=128, hidden_dim=128, n_heads=4, ffn_dim=512, state_kind="h"),
}

if MAMBA_SSM_AVAILABLE:
    MODEL_SPECS["mamba"] = dict(
        layers=3,
        emb_dim=128,
        hidden_dim=128,
        state_dim=16,
        expansion_factor=2,
        require_official_mamba=True,
        state_kind="h",
    )
elif RUN_MAMBA_LIKE_FALLBACK_WHEN_OFFICIAL_UNAVAILABLE:
    MODEL_SPECS["mamba_like"] = dict(
        layers=3,
        emb_dim=128,
        hidden_dim=128,
        state_dim=16,
        expansion_factor=2,
        state_kind="h",
    )

# Direct-run defaults. These are deliberately small.
SEEDS = [0]
TRAINING_STEPS = 5
BATCH_SIZE = 16
EVAL_EVERY = 5
EXTRACT_EXAMPLES = 64
EXTRACT_BATCH_SIZE = 32
LR = 3e-4

pd.DataFrame(DYCK_SETTINGS).T

,dyck_pairs,total_length,seq_len,repeat_prob,n_tasks,prefix_probe_max_len
dyck_no_noise,24.0,48.0,48.0,1.0,512.0,7.0
dyck_50_noise,10.0,20.0,48.0,0.5,512.0,7.0


## 2. Mamba Availability Check

If the official `mamba-ssm` package is installed, this notebook runs `model_name="mamba"` with `require_official_mamba=True`. On this Windows environment, official `mamba-ssm` is not installed, so the notebook can optionally run `model_name="mamba_like"` as a smoke-test fallback. Treat `mamba_like` as an engineering placeholder, not as official Mamba evidence.

In [4]:
mamba_ssm_available = importlib.util.find_spec("mamba_ssm") is not None
test_model_name = "mamba" if mamba_ssm_available else "mamba_like"
test_mamba = build_model(
    test_model_name,
    vocab_size=12,
    emb_dim=16,
    hidden_dim=16,
    layers=1,
    require_official_mamba=mamba_ssm_available,
)
x = torch.randint(0, 12, (2, 8))
with torch.no_grad():
    logits = test_mamba(x)
    states = test_mamba.extract_states(x)

{
    "mamba_ssm_available": mamba_ssm_available,
    "requested_model": test_model_name,
    "model_class": type(test_mamba).__name__,
    "uses_mamba_ssm": getattr(test_mamba, "uses_mamba_ssm", None),
    "logits_shape": tuple(logits.shape),
    "states_shape": tuple(states.shape),
}

{'mamba_ssm_available': False,
 'requested_model': 'mamba_like',
 'model_class': 'MambaLikeLM',
 'uses_mamba_ssm': False,
 'logits_shape': (2, 8, 12),
 'states_shape': (2, 8, 16)}

## 3. Train And Extract

In [5]:
def run_one(setting_name, setting_kwargs, model_name, model_spec, seed):
    run_dir = RESULTS / setting_name / f"{model_name}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    task_cfg = DyckConfig(**setting_kwargs, device=DEVICE)
    sampler = DyckSampler(task_cfg, seed=seed)
    model_kwargs = {k: v for k, v in model_spec.items() if k != "state_kind"}

    torch.manual_seed(seed)
    model = build_model(model_name, vocab_size=sampler.vocab_size, **model_kwargs).to(DEVICE)

    config_record = {
        "setting_name": setting_name,
        "task": {**setting_kwargs, "device": DEVICE},
        "model_name": model_name,
        "model": model_spec,
        "seed": seed,
        "training_steps": TRAINING_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "device": DEVICE,
    }
    save_json(config_record, run_dir / "config.json")

    train_log = train_causal_lm(
        model=model,
        sampler=sampler,
        steps=TRAINING_STEPS,
        batch_size=BATCH_SIZE,
        lr=LR,
        run_dir=run_dir,
        eval_every=EVAL_EVERY,
        device=DEVICE,
    )
    eval_metrics = evaluate_causal_lm(model=model, sampler=sampler, batch_size=BATCH_SIZE, device=DEVICE)
    save_json({"train": train_log, "eval": eval_metrics}, run_dir / "metrics.json")

    hidden, labels = extract_hidden_states(
        model=model,
        sampler=sampler,
        state_kind=model_spec.get("state_kind", "h"),
        layer=-1,
        num_examples=EXTRACT_EXAMPLES,
        batch_size=EXTRACT_BATCH_SIZE,
        max_prefix_len=task_cfg.prefix_probe_max_len,
        device=DEVICE,
        run_dir=run_dir,
    )
    return run_dir, eval_metrics, tuple(hidden.shape), labels.shape

In [6]:
run_rows = []

for setting_name, setting_kwargs in DYCK_SETTINGS.items():
    for model_name, model_spec in MODEL_SPECS.items():
        for seed in SEEDS:
            run_dir, eval_metrics, hidden_shape, labels_shape = run_one(
                setting_name, setting_kwargs, model_name, model_spec, seed
            )
            run_rows.append({
                "setting": setting_name,
                "model": model_name,
                "seed": seed,
                "run_dir": str(run_dir.relative_to(ROOT)),
                "loss": eval_metrics["loss"],
                "accuracy": eval_metrics["accuracy"],
                "dyck_accuracy": eval_metrics["dyck_accuracy"],
                "hidden_shape": hidden_shape,
                "labels_shape": labels_shape,
            })

runs_df = pd.DataFrame(run_rows)
runs_df.to_csv(RESULTS / "runs.csv", index=False)
runs_df


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



,setting,model,seed,run_dir,loss,accuracy,dyck_accuracy,hidden_shape,labels_shape
0,dyck_no_noise,rnn,0,results\dyck_all_models_notebook\dyck_no_noise...,1.270225,0.538564,0.538564,"(448, 256)","(448, 15)"
1,dyck_no_noise,lstm,0,results\dyck_all_models_notebook\dyck_no_noise...,2.340602,0.489362,0.489362,"(448, 128)","(448, 15)"
2,dyck_no_noise,transformer,0,results\dyck_all_models_notebook\dyck_no_noise...,0.832573,0.541223,0.541223,"(448, 128)","(448, 15)"
3,dyck_no_noise,mamba_like,0,results\dyck_all_models_notebook\dyck_no_noise...,0.963148,0.537234,0.537234,"(448, 128)","(448, 15)"
4,dyck_50_noise,rnn,0,results\dyck_all_models_notebook\dyck_50_noise...,2.339320,0.218085,0.525641,"(1159, 256)","(1159, 15)"
5,dyck_50_noise,lstm,0,results\dyck_all_models_notebook\dyck_50_noise...,2.454777,0.202128,0.487179,"(1159, 128)","(1159, 15)"
6,dyck_50_noise,transformer,0,results\dyck_all_models_notebook\dyck_50_noise...,2.484999,0.212766,0.506410,"(1159, 128)","(1159, 15)"
7,dyck_50_noise,mamba_like,0,results\dyck_all_models_notebook\dyck_50_noise...,2.443755,0.219415,0.525641,"(1159, 128)","(1159, 15)"


## 4. Probes And Compression

The probe layer uses sklearn if explicitly enabled with `HSE_USE_SKLEARN=1`; otherwise it uses the built-in torch/numpy fallback. This avoids local NumPy/scipy ABI issues.

In [7]:
def labels_path_for(run_dir):
    parquet = run_dir / "labels.parquet"
    csv = run_dir / "labels.csv"
    return parquet if parquet.exists() else csv


def analyze_one(run_dir, seed):
    X, labels = load_probe_data(run_dir / "hidden_states.pt", labels_path_for(run_dir))
    probe_results = run_sufficient_statistic_probes(X, labels, seed=seed)
    compression_rows, compression_summary = run_compression_probes(X, labels, seed=seed)

    probe_dir = run_dir / "probes"
    probe_dir.mkdir(exist_ok=True)
    compression_rows.to_csv(probe_dir / "compression_probe_rows.csv", index=False)

    summary = {
        "left_r2": probe_results["left"]["r2"],
        "right_r2": probe_results["right"]["r2"],
        "height_r2": probe_results["height"]["r2"],
        "height_mae": probe_results["height"]["mae"],
        **probe_results["geometry"],
        **compression_summary,
    }
    save_json(summary, probe_dir / "summary.json")
    return summary

In [8]:
analysis_rows = []

for row in run_rows:
    run_dir = ROOT / row["run_dir"]
    summary = analyze_one(run_dir, seed=row["seed"])
    analysis_rows.append({
        "setting": row["setting"],
        "model": row["model"],
        "seed": row["seed"],
        **summary,
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df.to_csv(RESULTS / "probe_summary.csv", index=False)
analysis_df

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

C:\Users\HP\Desktop\Research\UWM Yiqiao Zhong\Hidden State Evaluation\src\hse\analysis\probes\dyck.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hidden = torch.load(h

,setting,model,seed,left_r2,right_r2,height_r2,height_mae,cos_height_left_minus_right,cos_left_right,relevant_retention,irrelevant_decodability,irrelevant_forgetting
0,dyck_no_noise,rnn,0,0.999414,0.999401,0.997775,0.029887,1.0,-0.790123,0.981633,0.000000,1.000000
1,dyck_no_noise,lstm,0,0.999604,0.999763,0.999111,0.028469,1.0,-0.738577,0.994774,0.000000,1.000000
2,dyck_no_noise,transformer,0,0.994873,0.991929,0.975893,0.126841,1.0,-0.946707,0.985669,0.000000,1.000000
3,dyck_no_noise,mamba_like,0,0.998654,0.998375,0.995524,0.054598,1.0,-0.472833,0.984433,0.000000,1.000000
4,dyck_50_noise,rnn,0,0.721550,0.685385,0.532819,0.667228,1.0,0.324964,0.554387,0.373003,0.626997
5,dyck_50_noise,lstm,0,0.908214,0.921663,0.863550,0.349045,1.0,0.294987,0.837773,0.418620,0.581380
6,dyck_50_noise,transformer,0,0.822608,0.836146,0.655522,0.602827,1.0,-0.013012,0.721474,0.307633,0.692367
7,dyck_50_noise,mamba_like,0,0.880349,0.858213,0.795898,0.405757,1.0,0.231094,0.758941,0.408340,0.591660


## 5. Interpretation Checklist

For the smoke run, the numbers only verify that the pipeline runs. They are not research evidence. For the full run, check:

- `dyck_accuracy`: task performance on Dyck target positions.
- `height_r2`: linear decodability of stack height.
- `cos_height_left_minus_right`: whether the learned height direction matches `w_left - w_right`.
- `relevant_retention`: how much sufficient-statistic information is retained.
- `irrelevant_forgetting`: how much noise/prefix detail is discarded.

A strong compression story means high relevant retention and high irrelevant forgetting, not just low information overall.